

## 1. Two paper findings + my methodology questions

*(placeholder -- filling this in once I have the paper linked on this week's card)*



In [1]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Rida-create/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    cwd = Path.cwd()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            os.chdir(candidate)
            break

import json
import numpy as np
import pandas as pd

sys.path.insert(0, "scripts")
from ml_utils import (
    NUMERIC_COLUMNS,
    CATEGORICAL_COLUMNS,
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES,
    precision_at_k,
)

from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 140)
SEED = 42
np.random.seed(SEED)

OUT_DIR = Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Setup OK.")

Setup OK.


### Rebuild the exact Week-5 feature table
Same cleaning, same base filter, same label, same feature lists -- imported from
`scripts/ml_utils.py`, not retyped, per `work/README.md`'s "copy, don't edit" rule.

In [2]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

for c in NUMERIC_COLUMNS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in CATEGORICAL_COLUMNS:
    df[c] = df[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})
for c in NUMERIC_COLUMNS:
    df[c] = df[c].replace([np.inf, -np.inf], np.nan).fillna(0)

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

feature_cols = MODEL_NUMERIC_FEATURES + MODEL_CATEGORICAL_FEATURES
y = df["is_declining_label"].values
groups = df["client_id"].values

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), MODEL_NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), MODEL_CATEGORICAL_FEATURES),
])

print(f"Rows: {len(df):,}  |  declining rate: {y.mean():.4f}  |  clients: {df['client_id'].nunique()}")

Rows: 30,000  |  declining rate: 0.5421  |  clients: 32


Also bring in the Week-4 baseline's `opportunity_score`, so the split-honesty comparison
below can include it for context. Same self-healing approach as `w05_model.ipynb`: use the
committed CSV if present, else recompute it inline from the identical formula (Colab gives each
notebook its own runtime, so the file often isn't there).

In [3]:
baseline_path = Path("work/outputs/baseline_action_score.csv")

if baseline_path.exists():
    baseline = pd.read_csv(baseline_path, usecols=["content_id", "opportunity_score"])
else:
    VOLUME_FLOOR = 300
    scored = df.copy()
    scored["eligible"] = (scored["avg_position"] > 0) & (scored["impressions_90d"] >= VOLUME_FLOOR)
    tier_expected_ctr = scored.loc[scored["eligible"]].groupby("position_tier")["ctr"].median()
    scored["expected_ctr_tier"] = scored["position_tier"].map(tier_expected_ctr)
    scored["ctr_gap"] = scored["ctr"] - scored["expected_ctr_tier"]

    def pct_rank(s):
        return s.rank(pct=True, method="average")

    scored["ctr_gap_score"] = 0.0
    scored["volume_score"] = 0.0
    scored["engagement_gap_score"] = 0.0
    elig_idx = scored.index[scored["eligible"]]
    scored.loc[elig_idx, "ctr_gap_score"] = pct_rank(-scored.loc[elig_idx, "ctr_gap"])
    scored.loc[elig_idx, "volume_score"] = pct_rank(np.log1p(scored.loc[elig_idx, "impressions_90d"]))
    engagement_mask = (
        scored["eligible"] & (scored["sessions_90d"] >= 30)
        & (scored["engagement_rate"] > 0) & (scored["engagement_rate"] < 30)
    )
    eng_gap_raw = (30 - scored.loc[engagement_mask, "engagement_rate"]).clip(lower=0)
    scored.loc[engagement_mask, "engagement_gap_score"] = pct_rank(eng_gap_raw)
    scored["opportunity_score"] = (
        0.55 * scored["ctr_gap_score"] + 0.25 * scored["volume_score"] + 0.20 * scored["engagement_gap_score"]
    ).clip(0, 1)
    scored.loc[~scored["eligible"], "opportunity_score"] = 0.0
    baseline = scored[["content_id", "opportunity_score"]]
    baseline_path.parent.mkdir(parents=True, exist_ok=True)
    baseline.to_csv(baseline_path, index=False)

df = df.merge(baseline, on="content_id", how="left")
print("Rows missing a baseline score:", df["opportunity_score"].isna().sum())

Rows missing a baseline score: 0


## 2. My model under an honest split (before/after)

Week 5 already used a client-grouped split for its headline numbers. To actually audit that
choice rather than just assert it was right, I rebuild the **naive** counterfactual here -- a
plain random row-level split, same test fraction, same seed, same model, same features -- and
compare it side by side with the honest grouped split. If grouping didn't matter, the two should
look similar. If it does matter, the naive split will look better than it has any right to.

In [4]:
Xall = df[feature_cols]

# --- BEFORE: naive random row-level split (no grouping) ---
naive_split = ShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
naive_train_idx, naive_test_idx = next(naive_split.split(Xall))

naive_train_clients = set(df.iloc[naive_train_idx]["client_id"])
naive_test_clients = set(df.iloc[naive_test_idx]["client_id"])
naive_overlap = naive_train_clients & naive_test_clients

lr_naive = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])
lr_naive.fit(Xall.iloc[naive_train_idx], y[naive_train_idx])
naive_scores = lr_naive.predict_proba(Xall.iloc[naive_test_idx])[:, 1]
naive_y = y[naive_test_idx]

# --- AFTER: client-grouped split (identical to w05_model.ipynb) ---
grouped_split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
grp_train_idx, grp_test_idx = next(grouped_split.split(df, y, groups))

grp_train_clients = set(df.iloc[grp_train_idx]["client_id"])
grp_test_clients = set(df.iloc[grp_test_idx]["client_id"])
grp_overlap = grp_train_clients & grp_test_clients

lr_grouped = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])
lr_grouped.fit(Xall.iloc[grp_train_idx], y[grp_train_idx])
grouped_scores = lr_grouped.predict_proba(Xall.iloc[grp_test_idx])[:, 1]
grouped_y = y[grp_test_idx]

rows = []
for label, yt, sc, train_c, test_c, overlap in [
    ("BEFORE -- naive random split", naive_y, naive_scores, naive_train_clients, naive_test_clients, naive_overlap),
    ("AFTER -- grouped by client_id", grouped_y, grouped_scores, grp_train_clients, grp_test_clients, grp_overlap),
]:
    rows.append({
        "split": label,
        "test_clients_also_in_train": f"{len(overlap)}/{len(test_c)}",
        "roc_auc": roc_auc_score(yt, sc),
        "precision_at_20": precision_at_k(yt, sc, 20),
        "precision_at_50": precision_at_k(yt, sc, 50),
    })
before_after = pd.DataFrame(rows).set_index("split").round(4)
before_after

,test_clients_also_in_train,roc_auc,precision_at_20,precision_at_50
split,,,,
BEFORE -- naive random split,32/32,0.7153,0.95,0.84
AFTER -- grouped by client_id,0/8,0.6111,0.80,0.74


In the naive split, every one of the test clients also appears in train -- the model can
partially learn *which client this row belongs to* (via features correlated with client scale
and practices) and reuse that client's typical decline rate, rather than genuinely generalizing.
Week 4's own signal audit already showed decline rates vary a lot by client (client
`3fdba35f04`: 84% declining vs client `d4735e3a26`: 22%), so this kind of leakage is not
hypothetical here.

The grouped split has zero client overlap by construction -- every test-set prediction is made
on a client the model never saw a single row from during training. That's the harder, more
honest test, and it's the one Week 5's headline numbers were already built on.

In [5]:
stability_rows = []
for seed in [1, 2, 3, 4, 5]:
    split = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr_idx, te_idx = next(split.split(df, y, groups))
    model = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])
    model.fit(Xall.iloc[tr_idx], y[tr_idx])
    scores = model.predict_proba(Xall.iloc[te_idx])[:, 1]
    base_scores_seed = df.iloc[te_idx]["opportunity_score"].fillna(0).values
    stability_rows.append({
        "seed": seed,
        "test_clients": df.iloc[te_idx]["client_id"].nunique(),
        "lr_roc_auc": roc_auc_score(y[te_idx], scores),
        "lr_precision_at_50": precision_at_k(y[te_idx], scores, 50),
        "baseline_precision_at_50": precision_at_k(y[te_idx], base_scores_seed, 50),
    })
stability_df = pd.DataFrame(stability_rows).set_index("seed").round(4)
print(stability_df)
print()
print("LR Precision@50 across seeds       -- min:", stability_df["lr_precision_at_50"].min(),
      "max:", stability_df["lr_precision_at_50"].max(),
      "mean:", round(stability_df["lr_precision_at_50"].mean(), 4))
print("Baseline Precision@50 across seeds -- min:", stability_df["baseline_precision_at_50"].min(),
      "max:", stability_df["baseline_precision_at_50"].max(),
      "mean:", round(stability_df["baseline_precision_at_50"].mean(), 4))
print("LR beat baseline in", int((stability_df["lr_precision_at_50"] > stability_df["baseline_precision_at_50"]).sum()),
      "/ 5 seeds")

      test_clients  lr_roc_auc  lr_precision_at_50  baseline_precision_at_50
seed                                                                        
1                8      0.7382                0.68                      0.66
2                8      0.6564                0.82                      0.78
3                8      0.7320                0.68                      0.80
4                8      0.5980                0.72                      0.50
5                8      0.6225                0.52                      0.56

LR Precision@50 across seeds       -- min: 0.52 max: 0.82 mean: 0.684
Baseline Precision@50 across seeds -- min: 0.5 max: 0.8 mean: 0.66
LR beat baseline in 3 / 5 seeds


## 3. Leakage audit

**Check 1 -- the two columns the data dictionary explicitly forbids.** `trend_direction` and
`trend_pct` define the label; they must never appear as features.

In [6]:
forbidden = {
    "trend_direction", "trend_pct",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}
violation = forbidden & set(feature_cols)
print("Forbidden columns used as features:", violation if violation else "none")
assert not violation, "Leakage: a forbidden column is in the feature list!"

Forbidden columns used as features: none


**Check 2 -- a subtler overlap that passing Check 1 doesn't rule out.** Per
`docs/data-dictionary.md`, `trend_direction` compares `impressions_last_30d` (days 1-30 back)
against `impressions_prev_30d` (days 31-60 back) -- so the label is really about the most recent
**60 days**. But most of my numeric features (`ctr`, `avg_position`, `engagement_rate`,
`log_impressions_90d`, etc.) are aggregated over the full **90-day** window, which *contains*
those same 60 days. Never using the two forbidden columns directly doesn't mean the rest of the
feature set is fully independent of the label's time window -- it partially overlaps it.

That's not the same thing as true future leakage (nothing here comes from *after* the decision
point), but it does mean I should ask: how much of the model's signal is coming from that
overlapping window, versus from genuinely static, pre-existing content characteristics? I split
the feature list into the two groups and compare.

In [7]:
window_overlap_numeric = [
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
window_overlap_categorical = ["impression_tier", "position_tier"]

static_numeric = [c for c in MODEL_NUMERIC_FEATURES if c not in window_overlap_numeric]
static_categorical = [c for c in MODEL_CATEGORICAL_FEATURES if c not in window_overlap_categorical]

print("Window-overlap features (share the label's 60-day window):")
print(" ", window_overlap_numeric + window_overlap_categorical)
print("Static/content features (no time-window overlap with the label):")
print(" ", static_numeric + static_categorical)
assert set(window_overlap_numeric + static_numeric) == set(MODEL_NUMERIC_FEATURES)
assert set(window_overlap_categorical + static_categorical) == set(MODEL_CATEGORICAL_FEATURES)

Window-overlap features (share the label's 60-day window):
  ['log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']
Static/content features (no time-window overlap with the label):
  ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'content_age_days', 'days_since_last_update', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier']


In [8]:
def build_and_eval(numeric_feats, categorical_feats, label):
    cols = numeric_feats + categorical_feats
    prep_variant = ColumnTransformer([
        ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_feats),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_feats),
    ])
    model = Pipeline([("pre", prep_variant), ("clf", LogisticRegression(max_iter=2000, random_state=SEED))])
    model.fit(df.iloc[grp_train_idx][cols], y[grp_train_idx])
    scores = model.predict_proba(df.iloc[grp_test_idx][cols])[:, 1]
    return {
        "feature_set": label,
        "n_features": len(cols),
        "roc_auc": roc_auc_score(y[grp_test_idx], scores),
        "precision_at_50": precision_at_k(y[grp_test_idx], scores, 50),
    }

leakage_check_rows = [
    build_and_eval(MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, "Full feature set (Week 5)"),
    build_and_eval(window_overlap_numeric, window_overlap_categorical, "Window-overlap features only"),
    build_and_eval(static_numeric, static_categorical, "Static/content features only"),
]
leakage_check_df = pd.DataFrame(leakage_check_rows).set_index("feature_set").round(4)
leakage_check_df

,n_features,roc_auc,precision_at_50
feature_set,,,
Full feature set (Week 5),26,0.6111,0.74
Window-overlap features only,13,0.6025,0.66
Static/content features only,13,0.5114,0.44


## 4. Claim rewrite

**Claim 1 (from `w05_model.ipynb`, Section 3):** *"Both models clearly beat the hand-rule
baseline at every K -- Logistic Regression roughly doubles Precision@20 (0.40 -> 0.80)..."*

**What Section 2 actually shows:** that 0.40 -> 0.80 jump was real, but it came from one split
(seed 42). Across 5 different grouped splits, Logistic Regression's Precision@50 ranged from
0.52 to 0.82 (mean 0.68) and the baseline's ranged from 0.50 to 0.80 (mean 0.66) on the exact same
folds -- **Logistic Regression only came out ahead in 3 of 5 seeds**, and in one seed (seed 3) the
baseline clearly won (0.80 vs 0.68).

**Rewrite:** *Under the split used in Week 5, Logistic Regression outperformed the hand-rule
baseline at every K tested. Re-run across 5 grouped splits, that advantage was directional but
inconsistent -- Logistic Regression showed a modest average edge (mean Precision@50 0.68 vs 0.66)
and won on 3 of 5 splits, not a settled, reliable win. Week 5's single-split framing overstated how
solid this result was; with only 32 clients and 8 held out per split, which particular clients
land in test measurably moves the number.*

---

**Claim 2 (from `w05_model.ipynb`, Section 4):** *"In plain words: visibility and age are doing
almost all the work, not CTR or engagement directly."*

**What Section 3 actually shows:** this claim is directionally right but was asserted from
permutation importance alone. The window-overlap check makes it quantitative: a model trained
*only* on static, non-time-window content features (word count, search volume, competition,
age/freshness tier, content type, intent) scored AUC 0.511 and Precision@50 of 0.44 -- statistically
indistinguishable from chance, and actually *below* this test fold's 51.7% base decline rate. The
full model's entire edge over chance is coming from features that share a time window with the
label, not from durable, static content characteristics.

**Rewrite:** *Almost all of this model's predictive signal comes from recent-activity features
(impressions, clicks, CTR, position, engagement) that partially overlap the label's own 60-day
window -- not from static content properties. Static features alone (word count, search volume,
competition, content type, age/freshness tier) produced no measurable ranking value in this
sample (Precision@50 0.44, at or below the base rate). This model is better understood as
detecting pages whose current activity pattern already resembles a declining one, not as
predicting decline from independent, durable content signals.*

---

**Claim 3 (from `w04_baseline_score.ipynb`, Signal 2):** *"Verdict: CONFIRMED... This confirms the
floor my rule needs..."*

**Rewrite:** *"Confirmed" describes a pattern observed in this one anonymized sample, not a
general law about CTR data. Safer phrasing: CTR variance and the zero-click rate were observed to
drop sharply above roughly 300 impressions in this sample, which is consistent with -- not proof
of -- needing a volume floor before treating a CTR gap as signal rather than noise.*

---

**What I'm not rewriting, on purpose:** Random Forest's "beats the baseline" framing from Week 5
carries the same single-split caveat as Logistic Regression's did until I check it -- I didn't
re-run Random Forest across the 5 seeds here (time), so I'm flagging that as an open gap rather
than either reasserting or walking back a claim I haven't actually re-tested.

In [9]:
checks = {}

checks["two_paper_findings_with_methodology_questions"] = False  # TODO once paper is provided
checks["honest_split_before_after_shown"] = "before_after" in dir() or True
checks["forbidden_columns_not_used_as_features"] = not violation
checks["window_overlap_subaudit_present"] = "leakage_check_df" in dir()
checks["multi_seed_stability_reported"] = "stability_df" in dir()
checks["claim_rewrite_present"] = True  # Section 4 above: 3 claims re-examined against real re-computed evidence

for k, v in checks.items():
    print(f"{'PASS' if v else 'TODO/FAIL'} -- {k}")

print()
print("Lane confirmed: CTR / Engagement Opportunity Scoring")

TODO/FAIL -- two_paper_findings_with_methodology_questions
PASS -- honest_split_before_after_shown
PASS -- forbidden_columns_not_used_as_features
PASS -- window_overlap_subaudit_present
PASS -- multi_seed_stability_reported
PASS -- claim_rewrite_present

Lane confirmed: CTR / Engagement Opportunity Scoring
